In [1]:
from huggingface_hub import login
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

In [2]:
load_dotenv()

True

In [3]:
login()

In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
q1 = "Can I still join the course after the start date ?"
v1 = model.encode(q1)

In [6]:
print(v1.shape)

(384,)


In [7]:
d = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [8]:
v1.dot(dv)

np.float32(0.323324)

In [9]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [10]:
v2.dot(dv)

np.float32(0.019730514)

## Embedding documents

In [11]:
from ingest import load_faq_data

In [12]:
documents = load_faq_data()

In [13]:
documents[3]

{'id': '52217fc51b',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: I have registered for the Data Engineering Bootcamp. When can I expect to receive the confirmation email?',
 'answer': "You don't need a confirmation email. You're accepted. You can start learning and submitting homework without registering. Registration was just to gauge interest before the start date."}

In [14]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [15]:
len(texts)

1346

In [16]:
from tqdm.auto import tqdm

In [17]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

In [18]:
len(vectors)

1346

In [19]:
import numpy as np

In [20]:
X = np.array(vectors)

In [21]:
X.shape

(1346, 384)

In [22]:
X.ndim

2

In [23]:
v1.shape

(384,)

In [24]:
scores = X.dot(v1)

In [25]:
top_score = np.argmax(scores)

In [26]:
top5 = np.argsort(scores)[-5:]

In [27]:
top5

array([  7, 536, 903, 621,   2])

In [28]:
np.argsort(-scores)[:5]

array([  2, 621, 903, 536,   7])

In [29]:
for ind in top5:
    print(scores[ind])
    print(documents[ind])
    print("\n")

0.56009996
{'id': '068529125b', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I follow the course after it finishes?', 'answer': 'Yes, we will keep all the materials available, so you can follow the course at your own pace after it finishes.\n\nYou can also continue reviewing the homeworks and prepare for the next cohort. You can also start working on your final capstone project.'}


0.6536312
{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}


0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start da

## Vector search with minsearch

In [30]:
from minsearch import VectorSearch

In [31]:
vindex = VectorSearch(
    keyword_fields=['course']
)

In [32]:
vindex.fit(X, documents)

In [33]:
query = "I just discovered the course. Can I still joint it ?"
q_v = model.encode(query)

In [34]:
vindex.search(
    q_v,
    num_results=5,
    filter_dict={"course": "llm-zoomcamp"}
)

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the for

In [35]:
import anthropic
from rag_helper import RAGVector

In [36]:
client = anthropic.Anthropic()

In [37]:
??RAGVector.llm

Signature: RAGVector.llm(self, prompt)
Docstring: <no docstring>
Source:   
    def llm(self, prompt):
        input_message = [
            #{"role": "developer", "content": self.instructions}, # Use this role if you use openai client
            {"role": "user", "content": prompt}
        ]

        response = self.llm_client.messages.create( # replace messages method by responses if you use openai client 
            model=self.model,
            system=self.instructions,
            max_tokens=1000,
            messages=input_message # replace messages attribut by input if you use openai client
        )
        return response
File:      ~/Documents/Learning/llm-zoomcamp-code/rag_helper.py
Type:      function

### I've already documents, X from vectors and vindex

In [38]:
vector_assistant = RAGVector(
    index=vindex,
    embedder=model,
    llm_client=client,
    model="claude-haiku-4-5"
)

In [39]:
result = vector_assistant.rag(query="How do I get a certificate ?")

In [42]:
print(result.content[0].text)

Based on the context provided, here's how to get a certificate:

## Requirements to Get a Certificate:

1. **Enroll in a "live" cohort** - You can only get a certificate if you complete the course with a live cohort. Self-paced mode does not award certificates.

2. **Submit your Capstone project** - You must pass the Capstone project to receive the certificate. While homework is recommended for reinforcement, it is not mandatory for certification.

3. **Complete peer reviews** - After submitting your project, you need to peer-review 3 capstones. You can only do this while the course is running and the peer-review list is compiled.

4. **Submit your project on time** - Make sure to submit your project while submissions are still being accepted during the cohort.

## Certificate Name:

When setting up your account, you can edit your course profile to specify the official name that will appear on your certificate. This should match your identification documents (passport, national ID card